#### import deps

In [ ]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


### Download example reference data point from langsmith

In [ ]:
from dotenv import load_dotenv #import 
import os 

load_dotenv("../../.env")
ls_client = Client()

In [ ]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)
print(dataset)

In [ ]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

In [ ]:
data_set = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))
fifteen_example = data_set[15]


In [ ]:
reference_input = fifteen_example.inputs
reference_output = fifteen_example.outputs



### RAG pipeline 

In [ ]:

qdrant_client = QdrantClient( # when we are running inside of docker network 
    url="http://localhost:6333"
)

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="minimal"
    )
    return response.choices[0].message.content

def rag_pipeline(question, topk_k=5):
    retrievedContext = retrieve_data(question, k=topk_k)
    processed_context = process_context(retrievedContext)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt) ## llm call w/ prompt
    
    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrievedContext["retrieved_context_ids"], ## surface to use for evaluation of precision and recall...
        "retrieved_context": retrievedContext["retrieved_context"],
        "similarity_scores": retrievedContext["similarity_scores"]
    }
    
    return final_answer

In [ ]:
rag_pipeline("Can I get a charger?")

### RAGAS METRICS

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy




In [ ]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

In [ ]:
reference_input

In [ ]:
reference_output

In [ ]:
result = rag_pipeline(reference_input['question'])


In [ ]:
result

In [ ]:
async def ragas_context_prescision_id_based(run, example):
    sample = SingleTurnSample(retrieved_context_ids=run["retrieved_context_ids"], reference_context_ids=example["reference_context_ids"])
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [ ]:
#signal vs noise ratio
await ragas_context_prescision_id_based(result, reference_output)

In [ ]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(retrieved_context_ids=run["retrieved_context_ids"], reference_context_ids=example["reference_context_ids"])
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_output)

In [ ]:
async def ragas_faithfulness(run):
    sample = SingleTurnSample(user_input=run["question"], response=run["answer"], retrieved_contexts=run["retrieved_context"])
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_faithfulness(result)

In [ ]:
async def ragas_relevancy(run):
    sample = SingleTurnSample(user_input=run["question"], response=run["answer"], retrieved_contexts=run["retrieved_context"])
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_relevancy(result)